# Weekend 2 — Inspect the model

**Goal:** decide whether the model you just trained is actually any good, using the
only tool you have this weekend — your eyes. Weekend 3 replaces this with a number.

By the end you should be able to answer:

1. Do the neighbours of a film look like that film? (The gate. If no, stop here.)
2. What exactly does shrinkage prevent? (Section 3 shows it failing without.)
3. Which movies can this model never recommend, and why?
4. Is it just recommending popular things with extra steps?

Run `python train.py` first.

---
## Setup

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import sys
sys.path.insert(0, "..")          # so we can reuse the functions from train.py

RAW = Path("../data/raw/ml-latest-small")
ARTIFACTS = Path("../artifacts")

d = np.load(ARTIFACTS / "model.npz", allow_pickle=False)
movie_ids = d["movie_ids"]
nbr_idx, nbr_sim = d["neighbor_idx"], d["neighbor_sim"]

cat = pd.read_csv(ARTIFACTS / "catalog.csv")
movies = pd.read_csv(RAW / "movies.csv").set_index("movieId")
ratings = pd.read_csv(RAW / "ratings.csv")
n_by_movie = ratings.groupby("movieId").size()

print(f"model    : {nbr_idx.shape[0]:,} movies x top-{nbr_idx.shape[1]} neighbours")
print(f"built    : lambda={float(d['lam']):g}  k={int(d['topk'])}  "
      f"center_m={float(d['center_m']):g}  global_mean={float(d['global_mean']):.4f}")
print(f"built at : {d['built_at']}")
print(f"catalog  : {len(cat):,} serveable movies")

The parameters travel *inside* the artifact. That is what lets you keep three
`.npz` files from weekend 3's sweep on disk and still know which is which.

`movie_ids` is the bridge: `nbr_idx` holds column positions, and `movie_ids[position]`
turns one back into a real MovieLens id.

---
## 1. The eyeball test

In [ ]:
pos_of = pd.Series(np.arange(len(movie_ids)), index=movie_ids)


def neighbors(query, n=10):
    """Top-n neighbours of the first movie whose title contains `query`."""
    hits = movies[movies.index.isin(movie_ids) & movies.title.str.contains(query, case=False, regex=False)]
    if hits.empty:
        raise LookupError(f"no rated movie matching {query!r}")
    movie_id = int(hits.index[0])
    i = int(pos_of[movie_id])
    ids = movie_ids[nbr_idx[i][:n]]
    out = pd.DataFrame({
        "sim": nbr_sim[i][:n].round(3),
        "n": n_by_movie.reindex(ids).values,
        "title": movies.loc[ids, "title"].values,
        "genres": movies.loc[ids, "genres"].values,
    })
    out.attrs["seed"] = f"{movies.loc[movie_id, 'title']}  [{movies.loc[movie_id, 'genres']}]  n={n_by_movie[movie_id]}"
    return out


def show(query, n=8):
    out = neighbors(query, n)
    print(out.attrs["seed"])
    print(out.to_string(index=False))
    print()


for q in ["Toy Story (1995)", "Star Wars: Episode IV", "Alien (1979)",
          "Pulp Fiction", "Notting Hill", "Shawshank"]:
    show(q)

**What you are checking.** Not "would I watch these" — whether the list is
*coherent*. Toy Story's neighbours should be animated/children's films. Star Wars IV's
should be the other two originals, well clear of everything else. If they aren't, the
usual causes in order are: centering skipped, the co-rating counts built from the
centered values instead of the rating pattern, or rows normalised instead of columns.

**Also notice what is off.** *The Godfather* sits high in Star Wars IV's list, and
*Groundhog Day* in Alien's. Neither is a sci-fi neighbour — they are films that
everybody who rates a lot of films has seen. With 610 users, item-item CF partly
rediscovers "popular among people who rate things". Section 5 quantifies that. It is a
real limitation; say it out loud in an interview rather than hoping nobody scrolls.

---
## 2. What the neighbour lists cost you

Every movie keeps only its top 50 neighbours. Two consequences worth seeing.

In [ ]:
# (a) Truncation makes the similarity matrix ASYMMETRIC.
a = int(pos_of[1])                                   # Toy Story
b = int(nbr_idx[a][0])                               # its closest neighbour
print(f"{movies.loc[movie_ids[a], 'title']} -> {movies.loc[movie_ids[b], 'title']}: "
      f"rank 1")
back = np.where(nbr_idx[b] == a)[0]
print(f"and back again: rank {back[0] + 1}" if len(back) else "and back again: NOT in its top 50")

# (b) Some movies have no usable neighbours at all.
unreachable = np.where(nbr_sim[:, 0] <= 0)[0]
print(f"\nmovies whose best neighbour has similarity <= 0: {len(unreachable):,}")

full = (nbr_sim > 0).sum(axis=1)
print(f"movies with a full set of 50 positive neighbours: {(full == nbr_sim.shape[1]).mean():.0%}")
print(f"median positive neighbours per movie: {np.median(full):.0f}")

Asymmetry is expected: popular films appear in many top-50 lists, obscure ones in
few. It is not a bug to fix — but it does mean "similar to" is directional here, and
your `/movies/{id}/similar` endpoint should say so if anyone asks.

---
## 3. Why shrinkage exists ← the cell that earns this notebook

Section 1 looked at *popular* films, where shrinkage barely changes anything. The
failure it prevents lives in the long tail. We recompute one movie's similarities at
`lambda = 0` and `lambda = 10`, reusing the functions from `train.py`.

In [ ]:
from train import (build_index, build_matrices, shrunk_user_mean,
                   center_ratings, normalize_columns, apply_shrinkage)

mids, uids, row_of, col_of = build_index(ratings)
R, B = build_matrices(ratings, len(uids), len(mids), row_of, col_of)
agg = ratings.groupby("userId").rating.agg(["sum", "count"]).reindex(uids)
mu = shrunk_user_mean(agg["sum"].to_numpy(float), agg["count"].to_numpy(float),
                      float(d["center_m"]), float(d["global_mean"]))
Cn = normalize_columns(center_ratings(R, mu.astype(np.float32)))[0].tocsc()
Bc = B.tocsc()
LAM = float(d["lam"])


def similarity_columns(movie_id):
    """Raw cosine, co-rater counts and shrunk cosine against every other movie."""
    i = int(pos_of[movie_id])
    cos = (Cn[:, i].T @ Cn).toarray().ravel().astype(np.float64)
    n_co = (Bc[:, i].T @ Bc).toarray().ravel().astype(np.float64)
    shrunk = apply_shrinkage(cos.copy(), n_co, LAM)
    cos[i] = shrunk[i] = -np.inf          # a movie is always its own best match
    return cos, n_co, shrunk


def ablation(movie_id, n=8):
    cos, n_co, shrunk = similarity_columns(movie_id)
    rank_after = np.empty(len(shrunk), int)
    rank_after[np.argsort(-shrunk)] = np.arange(len(shrunk))
    rows = []
    for rank, j in enumerate(np.argsort(-cos)[:n], start=1):
        rows.append({
            "rank@0": rank,
            "cosine": round(cos[j], 3),
            "co_raters": int(n_co[j]),
            f"shrunk@{LAM:g}": round(shrunk[j], 3),
            "new_rank": rank_after[j] + 1,
            "title": movies.loc[int(movie_ids[j]), "title"][:40],
        })
    return pd.DataFrame(rows)


# Pick, deterministically, a long-tail film whose best match at lambda=0 rests on a
# single shared rater and does NOT survive shrinkage.
def misled_by_one_rater(movie_id):
    cos, n_co, shrunk = similarity_columns(movie_id)
    best_before = np.argsort(-cos)[0]
    return n_co[best_before] == 1 and best_before not in np.argsort(-shrunk)[:5]


victim = next(m for m in n_by_movie[(n_by_movie >= 2) & (n_by_movie <= 6)].index
              if misled_by_one_rater(int(m)))

print(f"{movies.loc[victim, 'title']}  -- only {n_by_movie[victim]} people rated it\n")
print("ranked by RAW cosine (what you get with no shrinkage):")
print(ablation(int(victim)).to_string(index=False))

print(f"\ntop 5 after shrinking with lambda = {LAM:g}:")
cos, n_co, shrunk = similarity_columns(int(victim))
for j in np.argsort(-shrunk)[:5]:
    print(f"   {shrunk[j]:+.3f}   co-raters={int(n_co[j])}   "
          f"{movies.loc[int(movie_ids[j]), 'title'][:44]}")

**Read the `co_raters` column, then the `new_rank` column.**

Without shrinkage the top of this list is a pile of films at a near-identical, very high
cosine — every one of them sharing exactly **one** rater with our film. That is not a
similarity, it is a coincidence with two decimal places. One person watched both; the
cosine has no way to know that is all the evidence there is.

`new_rank` shows where each of them lands once shrinkage is applied: somewhere in the
fifties and sixties. What rises to the top instead are pairs with two or three co-raters
— still thin evidence, but three times as much of it, and now ranked accordingly.

`n_co / (n_co + lambda)` says: *a similarity is worth only as much as the evidence under
it.* Same move as the shrunk user mean above, and as `shrunk_mean` in the catalog. One
idea, three uses — which is worth noticing, because "shrink an estimate toward a prior
when the sample is small" is the reusable thing here, not this particular formula.

**What shrinkage does not do** is remove the long tail from the lists. A top-50 list has
fifty slots and something has to fill them. Section 5 shows where those films end up.

---
## 4. What the model cannot do

In [ ]:
best = nbr_sim[:, 0]
n_rat = n_by_movie.reindex(movie_ids).to_numpy()

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].scatter(n_rat, best, s=4, alpha=0.2, color="steelblue")
ax[0].set(xscale="log", xlabel="# ratings the movie has", ylabel="best neighbour similarity",
          title="Confidence follows evidence")
ax[0].grid(alpha=0.3, which="both")

ax[1].hist(best, bins=50, color="seagreen")
ax[1].set(xlabel="best neighbour similarity", ylabel="movies",
          title="Most movies have a weak best match")
plt.tight_layout(); plt.show()

for lo, hi, label in [(1, 2, "1 rating"), (2, 5, "2-4 ratings"),
                      (5, 20, "5-19 ratings"), (20, 10**9, "20+ ratings")]:
    m = (n_rat >= lo) & (n_rat < hi)
    print(f"{label:<14} {m.sum():>5,} movies   median best-neighbour similarity {np.median(best[m]):.3f}")

no_ratings = 9742 - len(movie_ids)
print(f"\nAnd {no_ratings} movies in the catalog have NO ratings at all.")
print("They have no column in R, so collaborative filtering cannot reach them, ever.")
print("That is item cold-start. It is a property of the method, not a bug -- fixing it")
print("means adding content features (genre, year, cast), which is a hybrid model.")

The scatter is the honest summary of this model: **confidence tracks evidence.**
A film with 200 ratings gets a strong, meaningful nearest neighbour. A film with 2 gets
a weak one, correctly. Shrinkage is what produces that slope instead of a flat line of
spurious 1.0s.

---
## 5. Is it just recommending popular films?

The fair worry about any CF model on a small dataset. Compare the popularity of
recommended films against the catalog as a whole.

In [ ]:
nbr_n = n_by_movie.reindex(movie_ids[nbr_idx.ravel()]).to_numpy()

print(f"median ratings, all movies in the model : {np.median(n_rat):>6.0f}")
print(f"median ratings, movies that appear as a neighbour : {np.median(nbr_n):>6.0f}")

appearances = np.bincount(nbr_idx.ravel(), minlength=len(movie_ids))
print(f"\ndistinct movies appearing in at least one top-50 list: "
      f"{(appearances > 0).sum():,} of {len(movie_ids):,} ({(appearances > 0).mean():.0%})")
print(f"share of all neighbour slots taken by the top 1% most-rated movies: "
      f"{appearances[np.argsort(-n_rat)[:len(movie_ids)//100]].sum() / appearances.sum():.1%}")

# Where do the 1-rating films actually end up?
one_rating = (n_rat == 1)
in_list = one_rating[nbr_idx]
print(f"\nmovies with a single rating: {one_rating.sum():,} "
      f"({one_rating.mean():.0%} of the model)")
print(f"  share of ALL neighbour slots they occupy            : {in_list.mean():>5.1%}")
print(f"  share of the slots of movies with 20+ ratings       : {in_list[n_rat >= 20].mean():>5.1%}")
print(f"  median similarity when one of them does appear      : {np.median(nbr_sim[in_list]):>5.3f}")

top = np.argsort(-appearances)[:10]
print("\nmost-recommended films (how often each appears in someone's top 50):")
for j in top:
    mid = int(movie_ids[j])
    print(f"   {appearances[j]:>4}x  n={n_by_movie[mid]:<4} {movies.loc[mid, 'title'][:50]}")

**The long-tail films take about a third of all neighbour slots — and essentially
none of the slots belonging to films anyone has actually rated.** They sit at similarity
~0.09, in each other's lists, at rank 24 of 50. Shrinkage could not delete them (fifty
slots need filling) but it did push them below anything with real evidence behind it,
which is the outcome that matters: they can never outrank a genuine neighbour where a
genuine neighbour exists.

Note this is **coverage of the neighbour lists**, not of actual recommendations —
what a user finally sees depends on their ratings too, and weekend 3's Recall@20 is what
settles whether any of this beats simply ranking by popularity.

Which is exactly the point of weekend 3. Everything above is *plausibility*, and a
recommender that loses to most-popular produces equally plausible lists. You cannot tell
from this notebook. That is not a flaw in the notebook — it is why the metric exists.

---
## 6. Findings

Fill these in from your own output, then start weekend 3.

In [ ]:
print(f"""
WEEKEND 2 FINDINGS
------------------
model                    {nbr_idx.shape[0]:,} movies x top-{nbr_idx.shape[1]}
parameters               lambda={float(d['lam']):g}  center_m={float(d['center_m']):g}
serveable catalog        {len(cat):,}
eyeball test             Toy Story -> {movies.loc[int(movie_ids[nbr_idx[int(pos_of[1])][0]]), 'title']}   <- PASS/FAIL?
movies with no positive neighbour   {int((nbr_sim[:, 0] <= 0).sum()):,}
median best-neighbour similarity    {np.median(nbr_sim[:, 0]):.3f}
neighbour-list coverage             {(np.bincount(nbr_idx.ravel(), minlength=len(movie_ids)) > 0).mean():.0%} of the catalog

WHAT I STILL CANNOT SAY
-----------------------
Whether this beats ranking every movie by popularity. Nothing in this notebook
answers that, and the lists look reasonable either way.

CARRY INTO WEEKEND 3
--------------------
1. Chronological leave-one-out    (hold out each user's LAST rating by timestamp)
2. Recall@20, three ways          (model / shrunk-mean popularity / random)
3. catalog.shrunk_mean is ready   (the baseline is already computed, just rank by it)
4. Center held-out users the same (global_mean and center_m are in model.npz)
5. THEN tune lambda and k         (against the metric, never against these lists)
""")